In [ ]:
import pandas as pd
import numpy as np
import zipfile
import requests
import io
import os

In [ ]:
zip_path = "https://download.companieshouse.gov.uk/BasicCompanyDataAsOneFile-2026-02-01.zip"
response = requests.get(zip_path)
if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z: #Use io.BytesIO to turn the "downloaded bytes" into a "virtual file"
        print(z.namelist())
else:
    print(f"Failed to reach the site. Status code: {response.status_code}")

['home/bulkdata/scripts/temp/217/BasicCompanyDataAsOneFile-2026-02-01.csv']


In [ ]:
# 1. Download the ZIP into memory
print("Downloading zip...")
response = requests.get(zip_path)

if response.status_code == 200:
    # 2. Open the ZIP using BytesIO
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        csv_filename = z.namelist()[0]

        # 3. Use 'with' to open the internal CSV and pass to Pandas
        with z.open(csv_filename) as f:
            # We use chunksize here to process the DataFrame in pieces
            chunks = []
            reader = pd.read_csv(f, chunksize=100000, low_memory=False)
            for chunk in reader:
              chunks.append(chunk)
            df = pd.concat(chunks, ignore_index=True)
            del chunks
            del chunk
            del reader
            del response
        del f
    del z
else:
    print("Download failed.")

In [ ]:
df.head()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,08209948,NaN,NaN,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025
1,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,4385,11743365 - COMPANIES HOUSE DEFAULT ADDRESS,NaN,CARDIFF,NaN,NaN,CF14 8LH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025
2,!FE NETWORK LIMITED,16873705,NaN,NaN,C/O AACSL ACCOUNTANT LIMITED,1ST FLOOR NORTH WESTGATE HOUSE,"HARLOW, ESSEX",NaN,UNITED KINGDOM,CM20 1YS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/12/2026,NaN
3,!NFLECTION ADVISORY LIMITED,15073164,NaN,NaN,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025
4,!NFOGENIE LTD,13522064,NaN,NaN,71-75 SHELTON STREET,NaN,LONDON,GREATER LONDON,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03/08/2026,20/07/2025


In [ ]:
df.columns

Index(['CompanyName', ' CompanyNumber', 'RegAddress.CareOf',
       'RegAddress.POBox', 'RegAddress.AddressLine1',
       ' RegAddress.AddressLine2', 'RegAddress.PostTown', 'RegAddress.County',
       'RegAddress.Country', 'RegAddress.PostCode', 'CompanyCategory',
       'CompanyStatus', 'CountryOfOrigin', 'DissolutionDate',
       'IncorporationDate', 'Accounts.AccountRefDay',
       'Accounts.AccountRefMonth', 'Accounts.NextDueDate',
       'Accounts.LastMadeUpDate', 'Accounts.AccountCategory',
       'Returns.NextDueDate', 'Returns.LastMadeUpDate',
       'Mortgages.NumMortCharges', 'Mortgages.NumMortOutstanding',
       'Mortgages.NumMortPartSatisfied', 'Mortgages.NumMortSatisfied',
       'SICCode.SicText_1', 'SICCode.SicText_2', 'SICCode.SicText_3',
       'SICCode.SicText_4', 'LimitedPartnerships.NumGenPartners',
       'LimitedPartnerships.NumLimPartners', 'URI', 'PreviousName_1.CONDATE',
       ' PreviousName_1.CompanyName', ' PreviousName_2.CONDATE',
       ' PreviousName_2.C

In [ ]:
len(df)

5678623

In [ ]:
df.dtypes

,0
CompanyName,object
CompanyNumber,object
RegAddress.CareOf,object
RegAddress.POBox,object
RegAddress.AddressLine1,object
RegAddress.AddressLine2,object
RegAddress.PostTown,object
RegAddress.County,object
RegAddress.Country,object
RegAddress.PostCode,object


In [ ]:
df['SICCode.SicText_1'].unique()

array(['99999 - Dormant Company', '59112 - Video production activities',
       '86210 - General medical practice activities', ...,
       '2871 - Manufacture steel drums, similar containers',
       '1713 - Prepare & spin worsted-type fibres',
       '2231 - Reproduction of sound recording'], dtype=object)

In [ ]:
df['SICCode.SicText_1']

,SICCode.SicText_1
0,99999 - Dormant Company
1,59112 - Video production activities
2,86210 - General medical practice activities
3,70229 - Management consultancy activities othe...
4,58290 - Other software publishing
...,...
5678618,61200 - Wireless telecommunications activities
5678619,79120 - Tour operator activities
5678620,56101 - Licensed restaurants
5678621,93199 - Other sports activities


In [ ]:
import re

In [ ]:
def SICCode(x):
  if not isinstance(x, str):
        return None

    # Searches for the first sequence of digits in the string
  match = re.search(r'\d+', x)
  return match.group(0) if match else None

In [ ]:
df['SIC_Numeric'] = df['SICCode.SicText_1'].apply(SICCode)

In [ ]:
sic_list = ['56101', '56102', '56103', '56210', '56290', '56301', '56302']
hospitality_df = df[df['SIC_Numeric'].isin(sic_list)]

In [ ]:
len(hospitality_df)

246706

In [ ]:
hospitality_df.SIC_Numeric

,SIC_Numeric
17,56101
34,56101
46,56102
116,56103
117,56290
...,...
5678561,56101
5678563,56302
5678568,56101
5678600,56101


In [ ]:
hospitality_df['SICCode.SicText_1']

,SICCode.SicText_1
17,56101 - Licensed restaurants
34,56101 - Licensed restaurants
46,56102 - Unlicensed restaurants and cafes
116,56103 - Take-away food shops and mobile food s...
117,56290 - Other food services
...,...
5678561,56101 - Licensed restaurants
5678563,56302 - Public houses and bars
5678568,56101 - Licensed restaurants
5678600,56101 - Licensed restaurants


In [ ]:
hospitality_df.isna().sum()

,0
CompanyName,0
CompanyNumber,0
RegAddress.CareOf,245994
RegAddress.POBox,245542
RegAddress.AddressLine1,5
RegAddress.AddressLine2,127667
RegAddress.PostTown,486
RegAddress.County,189077
RegAddress.Country,24295
RegAddress.PostCode,57


In [ ]:
hospitality_df = hospitality_df.dropna(subset = ['RegAddress.PostCode'])

In [ ]:
len(hospitality_df)

246649

In [ ]:
hospitality_df.head()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric
17,"""A TASTE OF TUSCANY"" LTD",15761044,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17/06/2026,03/06/2025,56101
34,"""BELLE-VUE"" ENTERPRISES LIMITED",04427981,NaN,NaN,2 THE CRESCENT,NaN,WISBECH,CAMBRIDGESHIRE,UNITED KINGDOM,PE13 1EH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11/05/2026,27/04/2025,56101
46,"""COOL COFFEE LONDON"" LTD",14428804,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,HA8 8BN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01/11/2026,18/10/2025,56102
116,"""PARKS"" ICECREAM FRANCHISE LIMITED",06991687,NaN,NaN,83 ORCHARD AVENUE,NaN,BLACKPOOL,LANCASHIRE,NaN,FY4 2NY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025,56103
117,"""PARO'S"" ITALIAN FOOD LTD",12789824,NaN,NaN,FLAT 1 MARLOW COURT,COLINDEEP LANE,LONDON,NaN,ENGLAND,NW9 6EB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03/06/2025,20/05/2024,56290


In [ ]:
def clean_company_names(text):
    if not isinstance(text, str):
        return text

    # 1. Remove double quotes
    text = text.replace('"', '')

    # 2. Define suffixes to remove (case insensitive)
    # \b ensures we match whole words only (so we don't accidentally trim "London" to "Lon")
    # \.? matches an optional period (e.g., LTD. or LTD)
    # $ ensures we only remove them if they are at the end of the string
    suffix_pattern = r'\b(LTD|LIMITED|PLC|LLP|CORP|INC)\b\.?$'
    bracket_pattern = r'\(.*?\)|\[.*?\]|\{.*?\}'
    combined_pattern = f"{bracket_pattern}|{suffix_pattern}"

    # Perform the replacement (ignoring case)
    cleaned = re.sub(combined_pattern, '', text, flags=re.IGNORECASE).strip()

    # 3. Clean up any trailing commas or extra whitespace left behind
    return cleaned.strip().rstrip(',')

In [ ]:
hospitality_df['CompanyName'] = hospitality_df['CompanyName'].apply(clean_company_names)

In [ ]:
hospitality_df.head()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric
17,A TASTE OF TUSCANY,15761044,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17/06/2026,03/06/2025,56101
34,BELLE-VUE ENTERPRISES,04427981,NaN,NaN,2 THE CRESCENT,NaN,WISBECH,CAMBRIDGESHIRE,UNITED KINGDOM,PE13 1EH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11/05/2026,27/04/2025,56101
46,COOL COFFEE LONDON,14428804,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,HA8 8BN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01/11/2026,18/10/2025,56102
116,PARKS ICECREAM FRANCHISE,06991687,NaN,NaN,83 ORCHARD AVENUE,NaN,BLACKPOOL,LANCASHIRE,NaN,FY4 2NY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025,56103
117,PARO'S ITALIAN FOOD,12789824,NaN,NaN,FLAT 1 MARLOW COURT,COLINDEEP LANE,LONDON,NaN,ENGLAND,NW9 6EB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03/06/2025,20/05/2024,56290


In [ ]:
hospitality_df['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,203211
Active - Proposal to Strike off,34669
Liquidation,8574
In Administration,130
Voluntary Arrangement,33
Live but Receiver Manager on at least one charge,16
In Administration/Administrative Receiver,16


In [ ]:
ratings_df = pd.read_csv('https://safhrsprodstorage.blob.core.windows.net/opendatafileblobstorage/FHRS_All_en-GB.csv')

/tmp/ipython-input-47541/1773690209.py:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  ratings_df = pd.read_csv('https://safhrsprodstorage.blob.core.windows.net/opendatafileblobstorage/FHRS_All_en-GB.csv')


In [ ]:
ratings_df.head()

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessTypeID,FHRSID,BusinessName,BusinessType,ConfidenceInManagement,Hygiene,...,LocalAuthorityName,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural
0,72 Regent Street,NaN,Cambridge,Cambridgeshire,1,1472964,@72.China,Restaurant/Cafe/Canteen,5.0,10.0,...,Cambridge City,0.126139,False,CB2 1DP,2024-05-14,fhrs_4_en-GB,4,NaN,FHRS,5.0
1,84 Regent Street,NaN,Cambridge,Cambridgeshire,1,1472953,1 and 1 Rougamo Ltd,Restaurant/Cafe/Canteen,10.0,10.0,...,Cambridge City,0.126407,False,CB2 1DP,2024-04-17,fhrs_3_en-GB,3,NaN,FHRS,10.0
2,108 Cherry Hinton Road,NaN,Cambridge,Cambridgeshire,4613,1888757,108 cherry hinton mini market,Retailers - other,0.0,5.0,...,Cambridge City,0.141757,False,CB1 7AJ,2024-12-17,fhrs_4_en-GB,4,NaN,FHRS,10.0
3,2 King Street,NaN,Cambridge,Cambridgeshire,1,1908335,13 Weeks,Restaurant/Cafe/Canteen,NaN,NaN,...,Cambridge City,0.121618,False,CB1 1LN,NaN,fhrs_awaitinginspection_en-GB,AwaitingInspection,NaN,FHRS,NaN
4,196 Mill Road,NaN,Cambridge,Cambridgeshire,1,1472773,196,Restaurant/Cafe/Canteen,0.0,5.0,...,Cambridge City,0.144063,False,CB1 3NF,2026-02-25,fhrs_5_en-GB,5,NaN,FHRS,5.0


In [ ]:
len(ratings_df)

602832

In [ ]:
len(hospitality_df)

246649

In [ ]:
hospitality_df['RegAddress.PostCode'] = hospitality_df['RegAddress.PostCode'].str.replace(' ','',regex = False)

In [ ]:
hospitality_df['RegAddress.PostCode']

,RegAddress.PostCode
17,WC2H9JQ
34,PE131EH
46,HA88BN
116,FY42NY
117,NW96EB
...,...
5678561,SG141PT
5678563,PR74EX
5678568,ME144BX
5678600,RH164LL


In [ ]:
ratings_df['PostCode'] = ratings_df['PostCode'].str.replace(' ','',regex = False)

In [ ]:
ratings_df['PostCode']

,PostCode
0,CB21DP
1,CB21DP
2,CB17AJ
3,CB11LN
4,CB13NF
...,...
602827,HU91TA
602828,HU31AB
602829,HU95PX
602830,HU95QR


In [ ]:
def checkpost(x):
    x=x.upper()
    x = x.replace(' ','') #To deal with cases where the space between outcode and incode is missing
    count = len(x)
    flag = False
    if x =='GIR0AA': #Special case for Girobank in Bootle, which does not follow postcode rules
        flag = True
    else:
        if 5<=count<=7:
            x = x[:-3] + ' ' + x[-3:]
            outcode = x.split(' ')[0]
            incode = x.split(' ')[1]
            if 2<=len(outcode)<=4 and len(incode)==3 and incode[0].isnumeric() and incode[1].isalpha() and incode[2].isalpha() and outcode[0].isalpha() and outcode[0] not in ['Q','V','X'] and not any(value in ['C','I','K','M','O','V'] for value in incode):
                if len(outcode)==4 and outcode[1].isalpha() and outcode[2].isnumeric() and outcode[3].isalnum():
                    flag = True
                elif len(outcode)==3 and ((outcode[2].isnumeric() and outcode[1].isalpha()) or (outcode[1].isnumeric() and outcode[2].isalnum())):
                    flag = True
                elif len(outcode)==2 and outcode[1].isnumeric():
                    flag = True
    return flag

In [ ]:
hospitality_df['PostCode_Validity'] = hospitality_df['RegAddress.PostCode'].apply(checkpost)

In [ ]:
hospitality_df['PostCode_Validity'].value_counts()

,count
PostCode_Validity,
True,246631
False,18


In [ ]:
hospitality_df[hospitality_df['PostCode_Validity'] == False]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity
99339,A LITTLE BIT OF ISH,13559706,NaN,NaN,23 LADYBOWER LANE,NaN,POULTON-LE-FYLDE,LANCASHIRE,NaN,FY67FV,...,NaN,NaN,NaN,NaN,NaN,NaN,24/08/2025,10/08/2024,56102,False
226656,AGORA EASTBOURNE,12309644,NaN,NaN,45 SIDNEY ROAD,NaN,EASTBOURNE,NaN,NaN,BN228B6,...,NaN,NaN,NaN,NaN,NaN,NaN,25/11/2022,11/11/2021,56101,False
1073270,CHUNKS STEAK BOX,12147012,NaN,NaN,UNIT 1 CLAYCOURT BUILDING,CRAWFORD STREET,ROCHDALE,LANCASHIRE,NaN,0L165RS,...,NaN,NaN,NaN,NaN,NaN,NaN,21/08/2021,07/08/2020,56103,False
1516738,DRINX TOP,09888237,NaN,NaN,114 HAMLET COURT ROAD,NaN,WESTCLIFF ON SEA,ESSEX,UNITED KINGDOM,SSO7LP,...,NaN,NaN,NaN,NaN,NaN,NaN,08/12/2026,24/11/2025,56103,False
1632330,ELENA HOSPITALITY,12026051,NaN,NaN,31 SACKVILLE STREET,NaN,MANCHESTER,NaN,NaN,M13L2,...,NaN,NaN,NaN,NaN,NaN,NaN,13/06/2022,30/05/2021,56290,False
2962849,LEVANT BOURNEMOUTH,07651187,NaN,NaN,150-152 OLD CHRISTCHURCH ROAD,NaN,BOURNEMOUTH,DORSET,NaN,BH11ML,...,NaN,NaN,NaN,NaN,NaN,NaN,14/07/2021,30/06/2020,56101,False
3111982,M AND B BYRNE,14591092,NaN,NaN,LORD OF THE MANOR,FORUM WAY,CRAMLINGTON,NORTHUMBERLAND,ENGLAND,NE23(6YD,...,NaN,NaN,NaN,NaN,NaN,NaN,26/01/2026,12/01/2025,56302,False
3807709,PANT YR ARDD,13679968,NaN,NaN,PANT YR ARDD,TREGARTH,BANGOR,GWYNEDD,UNITED KINGDOM,LL574PC,...,NaN,NaN,NaN,NaN,NaN,NaN,04/11/2026,21/10/2025,56302,False
3916307,PILLMAN,11744205,NaN,NaN,40 LIBERTY STREET,NaN,LONDON,NaN,NaN,SW9DEE,...,NaN,NaN,NaN,NaN,NaN,NaN,10/02/2021,30/12/2019,56302,False
4184688,RED HOT WORLD BUFFET,12370653,NaN,NaN,4 PARKVIEW ROAD,STOURBRIDGE,ENGLAND,WEST MIDLANDS,NaN,DY9-8XD,...,NaN,NaN,NaN,NaN,NaN,NaN,03/01/2027,20/12/2025,56101,False


In [ ]:
hospitality_df.loc[hospitality_df['PostCode_Validity']==False,['CompanyName','RegAddress.PostCode','PostCode_Validity','SICCode.SicText_1','CompanyStatus']]

,CompanyName,RegAddress.PostCode,PostCode_Validity,SICCode.SicText_1,CompanyStatus
99339,A LITTLE BIT OF ISH,FY67FV,False,56102 - Unlicensed restaurants and cafes,Active - Proposal to Strike off
226656,AGORA EASTBOURNE,BN228B6,False,56101 - Licensed restaurants,Active - Proposal to Strike off
1073270,CHUNKS STEAK BOX,0L165RS,False,56103 - Take-away food shops and mobile food s...,Active - Proposal to Strike off
1516738,DRINX TOP,SSO7LP,False,56103 - Take-away food shops and mobile food s...,Active
1632330,ELENA HOSPITALITY,M13L2,False,56290 - Other food services,Active - Proposal to Strike off
2962849,LEVANT BOURNEMOUTH,BH11ML,False,56101 - Licensed restaurants,Active - Proposal to Strike off
3111982,M AND B BYRNE,NE23(6YD,False,56302 - Public houses and bars,Active
3807709,PANT YR ARDD,LL574PC,False,56302 - Public houses and bars,Active
3916307,PILLMAN,SW9DEE,False,56302 - Public houses and bars,Active - Proposal to Strike off
4184688,RED HOT WORLD BUFFET,DY9-8XD,False,56101 - Licensed restaurants,Active


In [ ]:
ratings_df.isna().sum()

,0
AddressLine1,188644
AddressLine2,187105
AddressLine3,209430
AddressLine4,350265
BusinessTypeID,0
FHRSID,0
BusinessName,0
BusinessType,0
ConfidenceInManagement,127888
Hygiene,127888


In [ ]:
ratings_df = ratings_df.dropna(subset = ['PostCode'])

In [ ]:
len(ratings_df)

506235

In [ ]:
ratings_df['PostCode_Validity'] = ratings_df['PostCode'].apply(checkpost)

In [ ]:
ratings_df.head()

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessTypeID,FHRSID,BusinessName,BusinessType,ConfidenceInManagement,Hygiene,...,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural,PostCode_Validity
0,72 Regent Street,NaN,Cambridge,Cambridgeshire,1,1472964,@72.China,Restaurant/Cafe/Canteen,5.0,10.0,...,0.126139,False,CB21DP,2024-05-14,fhrs_4_en-GB,4,NaN,FHRS,5.0,True
1,84 Regent Street,NaN,Cambridge,Cambridgeshire,1,1472953,1 and 1 Rougamo Ltd,Restaurant/Cafe/Canteen,10.0,10.0,...,0.126407,False,CB21DP,2024-04-17,fhrs_3_en-GB,3,NaN,FHRS,10.0,True
2,108 Cherry Hinton Road,NaN,Cambridge,Cambridgeshire,4613,1888757,108 cherry hinton mini market,Retailers - other,0.0,5.0,...,0.141757,False,CB17AJ,2024-12-17,fhrs_4_en-GB,4,NaN,FHRS,10.0,True
3,2 King Street,NaN,Cambridge,Cambridgeshire,1,1908335,13 Weeks,Restaurant/Cafe/Canteen,NaN,NaN,...,0.121618,False,CB11LN,NaN,fhrs_awaitinginspection_en-GB,AwaitingInspection,NaN,FHRS,NaN,True
4,196 Mill Road,NaN,Cambridge,Cambridgeshire,1,1472773,196,Restaurant/Cafe/Canteen,0.0,5.0,...,0.144063,False,CB13NF,2026-02-25,fhrs_5_en-GB,5,NaN,FHRS,5.0,True


In [ ]:
ratings_df['PostCode_Validity'].value_counts()

,count
PostCode_Validity,
True,505639
False,596


In [ ]:
ratings_df.loc[ratings_df['PostCode_Validity']==False,['BusinessName','PostCode','PostCode_Validity','BusinessType','RatingValue']]

,BusinessName,PostCode,PostCode_Validity,BusinessType,RatingValue
7271,Cornish Hall End Village Hall,CM74JC,False,Restaurant/Cafe/Canteen,Exempt
30226,Pull My Leg,NR31OLX,False,Mobile caterer,5
43649,Co-operative,MK43,False,Retailers - supermarkets/hypermarkets,5
45997,Roam Cafe & Bistro,BedsLU55,False,Restaurant/Cafe/Canteen,5
46091,Snack Shack,Beds.SG19,False,Mobile caterer,5
...,...,...,...,...,...
597299,FV Sweet Water Of Newlyn,DN3135Y,False,Farmers/growers,Exempt
600672,Westside Day Nursery and the Acorns Centre Cafe,HU94OP,False,Hospitals/Childcare/Caring Premises,5
601821,Morrisons,YO317UK,False,Retailers - supermarkets/hypermarkets,5
601925,Oki's Kebab,YO,False,Mobile caterer,5


In [ ]:
ratings_df['BusinessType'].value_counts()

,count
BusinessType,
Restaurant/Cafe/Canteen,132924
Retailers - other,103591
Takeaway/sandwich shop,60858
Pub/bar/nightclub,50364
Hospitals/Childcare/Caring Premises,39228
School/college/university,35289
Other catering premises,29994
Retailers - supermarkets/hypermarkets,15670
Hotel/bed & breakfast/guest house,14172


In [ ]:
hospitality_df = hospitality_df[hospitality_df['PostCode_Validity'] == True]

In [ ]:
ratings_df = ratings_df[ratings_df['PostCode_Validity'] == True]

In [ ]:
len(hospitality_df)

246631

In [ ]:
len(ratings_df)

505639

In [ ]:
hospitality_df['RegAddress.AddressLine1'] = hospitality_df['RegAddress.AddressLine1'].str.upper()
ratings_df['BusinessName'] = ratings_df['BusinessName'].str.upper()
ratings_df['AddressLine1'] = ratings_df['AddressLine1'].str.upper()

In [ ]:
ratings_df = ratings_df.dropna(subset = ['AddressLine1'])

In [ ]:
hospitality_df['RegAddress.AddressLine1'] = hospitality_df['RegAddress.AddressLine1'].str.strip().str.replace(r'\s+', ' ', regex=True)
ratings_df['AddressLine1'] = ratings_df['AddressLine1'].str.strip().str.replace(r'\s+', ' ', regex=True)
ratings_df['BusinessName'] = ratings_df['BusinessName'].str.strip().str.replace(r'\s+', ' ', regex=True)
hospitality_df['CompanyName'] = hospitality_df['CompanyName'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [ ]:
name_lookup = ratings_df.groupby('PostCode')['BusinessName'].apply(list).to_dict()
address_lookup = ratings_df.groupby('PostCode')['AddressLine1'].apply(list).to_dict()
businessid_lookup = ratings_df.groupby('PostCode')['FHRSID'].apply(list).to_dict()
# 3. Map the lists into a new column in df1
hospitality_df['Potential_matches_name'] = hospitality_df['RegAddress.PostCode'].map(name_lookup)
hospitality_df['Potential_matches_address'] = hospitality_df['RegAddress.PostCode'].map(address_lookup)
hospitality_df['Potential_matches_businessid'] = hospitality_df['RegAddress.PostCode'].map(businessid_lookup)
#
# 4. Replace NaN (where no postcode match was found) with an empty list
hospitality_df['Potential_matches_name'] = hospitality_df['Potential_matches_name'].apply(lambda d: d if isinstance(d, list) else [])
hospitality_df['Potential_matches_address'] = hospitality_df['Potential_matches_address'].apply(lambda d: d if isinstance(d, list) else [])
hospitality_df['Potential_matches_businessid'] = hospitality_df['Potential_matches_businessid'].apply(lambda d: d if isinstance(d, list) else [])

In [ ]:
hospitality_df['Potential_matches_name']

,Potential_matches_name
17,[MIA & BEN ORGANIC]
34,[]
46,"[ACTIHEALTH, ACTIKID, LITTLE LEADERS STONEGROV..."
116,[]
117,[]
...,...
5678561,[OISHII JAPANESE RESTAURANT]
5678563,"[RETREAT, SPINNERS ARMS]"
5678568,[]
5678600,"[FRANCISCO LOUNGE, HART COUNTRY STORES, MILLETS]"


In [ ]:
type(hospitality_df['Potential_matches_address'][17])

list

In [ ]:
hospitality_df.columns

Index(['CompanyName', ' CompanyNumber', 'RegAddress.CareOf',
       'RegAddress.POBox', 'RegAddress.AddressLine1',
       ' RegAddress.AddressLine2', 'RegAddress.PostTown', 'RegAddress.County',
       'RegAddress.Country', 'RegAddress.PostCode', 'CompanyCategory',
       'CompanyStatus', 'CountryOfOrigin', 'DissolutionDate',
       'IncorporationDate', 'Accounts.AccountRefDay',
       'Accounts.AccountRefMonth', 'Accounts.NextDueDate',
       'Accounts.LastMadeUpDate', 'Accounts.AccountCategory',
       'Returns.NextDueDate', 'Returns.LastMadeUpDate',
       'Mortgages.NumMortCharges', 'Mortgages.NumMortOutstanding',
       'Mortgages.NumMortPartSatisfied', 'Mortgages.NumMortSatisfied',
       'SICCode.SicText_1', 'SICCode.SicText_2', 'SICCode.SicText_3',
       'SICCode.SicText_4', 'LimitedPartnerships.NumGenPartners',
       'LimitedPartnerships.NumLimPartners', 'URI', 'PreviousName_1.CONDATE',
       ' PreviousName_1.CompanyName', ' PreviousName_2.CONDATE',
       ' PreviousName_2.C

In [ ]:
hospitality_df = hospitality_df[hospitality_df['Potential_matches_address'].map(len)>0]

In [ ]:
import requests

# The URL for the company's filing history
url = "https://find-and-update.company-information.service.gov.uk/company/11978556/filing-history"

# Custom headers to mimic a browser (essential for Companies House)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# 1. Fetch the page content
response = requests.get(url, headers=headers)

# 2. Use pandas to read the HTML table
# read_html returns a list of DataFrames; usually, the filing history is the first one
dfs = pd.read_html(response.text)

# 3. Select the filing history table (index 0)
df = dfs[0]

/tmp/ipython-input-47541/1058243099.py:16: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(response.text)


In [ ]:
df

,Date (document was filed at Companies House),Type,Description (of the document filed at Companies House),"View / Download (PDF file, link opens in new window)"
0,26 Nov 2025,LIQ03,Liquidators' statement of receipts and payment...,View PDF Liquidators' statement of receipts a...
1,25 Nov 2024,LIQ03,Liquidators' statement of receipts and payment...,View PDF Liquidators' statement of receipts a...
2,23 Nov 2023,LIQ03,Liquidators' statement of receipts and payment...,View PDF Liquidators' statement of receipts a...
3,30 Nov 2022,LIQ03,Liquidators' statement of receipts and payment...,View PDF Liquidators' statement of receipts a...
4,30 Nov 2021,LIQ03,Liquidators' statement of receipts and payment...,View PDF Liquidators' statement of receipts a...
5,05 Dec 2020,LIQ03,Liquidators' statement of receipts and payment...,View PDF Liquidators' statement of receipts a...
6,25 Nov 2019,AD01,Registered office address changed from Gainsbo...,View PDF Registered office address changed fr...
7,23 Nov 2019,LIQ02,Statement of affairs,View PDF Statement of affairs - link opens i...
8,23 Nov 2019,600,Appointment of a voluntary liquidator,View PDF Appointment of a voluntary liquidato...
9,23 Nov 2019,RESOLUTIONS,Resolutions LRESEX ‐ Extraordinary resolutio...,View PDF Resolutions LRESEX ‐ Extraordinary...


In [ ]:
hospitality_df['URI']

,URI
17,http://business.data.gov.uk/id/company/15761044
46,http://business.data.gov.uk/id/company/14428804
282,http://business.data.gov.uk/id/company/15236922
291,http://business.data.gov.uk/id/company/12911325
297,http://business.data.gov.uk/id/company/11943499
...,...
5678558,http://business.data.gov.uk/id/company/09782772
5678561,http://business.data.gov.uk/id/company/12186046
5678563,http://business.data.gov.uk/id/company/10982198
5678600,http://business.data.gov.uk/id/company/12327287


In [ ]:
len(hospitality_df)

131032

In [ ]:
import time

In [ ]:
API_KEY = "5f985a05-c2b5-434e-a85b-1ddac308a957"

company_number = "10695244"  # Example company
url = f"https://api.company-information.service.gov.uk/company/{company_number}/filing-history"

# --- THE REQUEST ---
# Requests library handles the Basic Auth by taking a tuple: (username, password)
# Companies House uses the API Key as the username and an empty string as the password.
response = requests.get(url, auth=(API_KEY, ''))

if response.status_code == 200:
    data = response.json()
    #print(data)
    filings = data.get('items', [])
    # # Create the DataFrame
    df = pd.DataFrame(filings)

    # # Display the first few rows
    # print(f"Successfully retrieved {len(df)} filings.")
    print(df[['date', 'type', 'description']].head())
else:
    print(f"Error: {response.status_code}")
    print(response.text)

         date   type                                        description
0  2025-09-08  LIQ03  liquidation-voluntary-statement-of-receipts-an...
1  2024-09-16  LIQ03  liquidation-voluntary-statement-of-receipts-an...
2  2023-07-18   AD01  change-registered-office-address-company-with-...
3  2023-07-18  LIQ02         liquidation-voluntary-statement-of-affairs
4  2023-07-18    600    liquidation-voluntary-appointment-of-liquidator


In [ ]:
df

,transaction_id,barcode,type,date,category,subcategory,description,description_values,pages,action_date,paper_filed,links,resolutions,associated_filings
0,MzQ3OTgyNTYzMmFkaXF6a2N4,YEAD61ZK,LIQ03,2025-09-08,insolvency,voluntary,liquidation-voluntary-statement-of-receipts-an...,{'brought_down_date': '2025-07-04'},14,2025-07-04,True,{'self': '/company/10695244/filing-history/MzQ...,NaN,NaN
1,MzQzNDk3NjQ5MGFkaXF6a2N4,YDB9ENKZ,LIQ03,2024-09-16,insolvency,voluntary,liquidation-voluntary-statement-of-receipts-an...,{'brought_down_date': '2024-07-04'},14,2024-07-04,True,{'self': '/company/10695244/filing-history/MzQ...,NaN,NaN
2,MzM4NTY1MjAyM2FkaXF6a2N4,YC79OHRU,AD01,2023-07-18,address,NaN,change-registered-office-address-company-with-...,"{'change_date': '2023-07-18', 'old_address': '...",2,2023-07-18,True,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN
3,MzM4NTY1MjE0MWFkaXF6a2N4,YC79OILF,LIQ02,2023-07-18,insolvency,voluntary,liquidation-voluntary-statement-of-affairs,NaN,9,NaN,True,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN
4,MzM4NTY1MjAyNGFkaXF6a2N4,YC79OHZS,600,2023-07-18,insolvency,voluntary,liquidation-voluntary-appointment-of-liquidator,NaN,3,NaN,True,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN
5,WUM3OU9JWFNhZGlxemtjeA,NaN,RESOLUTIONS,2023-07-18,resolution,NaN,resolution,{'description': 'Resolutions'},1,NaN,True,{'self': '/company/10695244/filing-history/WUM...,"[{'category': 'liquidation', 'description': 'l...",NaN
6,MzM4MzkxMTkyOWFkaXF6a2N4,NaN,DISS16(SOAS),2023-06-23,dissolution,NaN,dissolved-compulsory-strike-off-suspended,NaN,1,NaN,True,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN
7,MzM4MTk5MDMxNWFkaXF6a2N4,NaN,GAZ1,2023-06-13,gazette,NaN,gazette-notice-compulsory,NaN,1,NaN,True,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN
8,MzM2NDE2OTQwOGFkaXF6a2N4,XBJXAWHC,AA,2022-12-30,accounts,NaN,accounts-with-accounts-type-total-exemption-full,{'made_up_date': '2022-03-31'},5,2022-03-31,NaN,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN
9,MzMzNDI1MTkyOGFkaXF6a2N4,XB0R5OUP,CS01,2022-03-28,confirmation-statement,NaN,confirmation-statement-with-no-updates,{'made_up_date': '2022-03-27'},3,2022-03-27,NaN,{'self': '/company/10695244/filing-history/MzM...,NaN,NaN


In [ ]:
df['description'][2]

'change-registered-office-address-company-with-date-old-address-new-address'

In [ ]:
df['description_values'][2]['old_address']

'Dilkhush 1462 Pershore Road Birmingham B30 2NT England'

In [ ]:
hospitality_df.head()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid
17,A TASTE OF TUSCANY,15761044,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,WC2H9JQ,...,NaN,NaN,NaN,17/06/2026,03/06/2025,56101,True,[MIA & BEN ORGANIC],[GARDEN STUDIOS],[1251615]
46,COOL COFFEE LONDON,14428804,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,HA88BN,...,NaN,NaN,NaN,01/11/2026,18/10/2025,56102,True,"[ACTIHEALTH, ACTIKID, LITTLE LEADERS STONEGROV...","[COMMUNITY CENTRE 5 HAYLING WAY EDGWARE, COMMU...","[1722995, 1722996, 1087224, 1762881]"
282,&FEAST,15236922,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,SW139HH,...,NaN,NaN,NaN,31/10/2026,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ..."
291,&IFOOD,12911325,NaN,NaN,316 GROVE GREEN ROAD,NaN,LONDON,NaN,ENGLAND,E114EA,...,NaN,NaN,NaN,11/10/2026,27/09/2025,56103,True,"[HEATHCOTE & STAR, SNS LOCAL, THE DRAGON FLAME...","[344 GROVE GREEN ROAD, GROUND FLOOR 320 GROVE ...","[1034984, 1888447, 1621047, 1073962]"
297,&LKH,11943499,NaN,NaN,1 SOUTH PARADE,NaN,WESTON-SUPER-MARE,NORTH SOMERSET,UNITED KINGDOM,BS231JP,...,NaN,NaN,NaN,25/04/2026,11/04/2025,56302,True,[CORNISH CABIN (THE)],[ROYAL HOTEL],[1735331]


In [ ]:
!pip install thefuzz

In [ ]:
!pip install rapidfuzz

In [ ]:
from rapidfuzz import process, fuzz

In [ ]:
ratings_df.columns

Index(['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4',
       'BusinessTypeID', 'FHRSID', 'BusinessName', 'BusinessType',
       'ConfidenceInManagement', 'Hygiene', 'Latitude',
       'LocalAuthorityBusinessID', 'LocalAuthorityCode', 'LocalAuthorityName',
       'Longitude', 'NewRatingPending', 'PostCode', 'RatingDate', 'RatingKey',
       'RatingValue', 'RightToReply', 'SchemeType', 'Structural',
       'PostCode_Validity'],
      dtype='object')

In [ ]:
hospitality_df.loc[291]

,291
CompanyName,&IFOOD
CompanyNumber,12911325
RegAddress.CareOf,NaN
RegAddress.POBox,NaN
RegAddress.AddressLine1,316 GROVE GREEN ROAD
RegAddress.AddressLine2,NaN
RegAddress.PostTown,LONDON
RegAddress.County,NaN
RegAddress.Country,ENGLAND
RegAddress.PostCode,E114EA


In [ ]:
ratings_address = hospitality_df['Potential_matches_name'][282]
ratings_address

['& FEAST',
 'BARNES PANTRY',
 "GAIL'S BARNES",
 "NATSON'S",
 'THE GINGER PIG',
 'TWO PEAS IN A POD']

In [ ]:
hospitality_address = hospitality_df['CompanyName'][282]
hospitality_address

'&FEAST'

In [ ]:
all_scores = process.extract(hospitality_address, ratings_address, scorer=fuzz.token_set_ratio, limit=None)

In [ ]:
all_scores

[('& FEAST', 92.3076923076923, 0),
 ('TWO PEAS IN A POD', 34.78260869565217, 5),
 ('BARNES PANTRY', 31.578947368421055, 1),
 ("GAIL'S BARNES", 31.578947368421055, 2),
 ("NATSON'S", 28.57142857142857, 3),
 ('THE GINGER PIG', 20.0, 4)]

In [ ]:
all_scores[0]

('& FEAST', 92.3076923076923, 0)

In [ ]:
type(all_scores)

list

In [ ]:
hospitality_df['IncorporationDate'] = pd.to_datetime(hospitality_df['IncorporationDate'], format='%d/%m/%Y', errors='coerce')

In [ ]:
val = "MRE FISH & CHIPS"
search = ["ANN'S FLOWERS", 'MARINE FISH AND CHIPS', 'RODMILL POST OFFICE', 'THE CO-OPERATIVE GROUP', "UNCLE SAM'S", 'WINDMILLS BAKERY']
all_scores = process.extract(val, search, scorer=fuzz.token_set_ratio, limit=None)
print(all_scores)

[('MARINE FISH AND CHIPS', 81.08108108108108, 1), ('THE CO-OPERATIVE GROUP', 31.578947368421055, 3), ("UNCLE SAM'S", 29.629629629629633, 4), ('RODMILL POST OFFICE', 28.57142857142857, 2), ("ANN'S FLOWERS", 27.58620689655173, 0), ('WINDMILLS BAKERY', 25.0, 5)]


In [ ]:
def active_cases(row):
  status = row['CompanyStatus']
  hospitality_address = row['RegAddress.AddressLine1']
  hospitality_name = row['CompanyName']
  ratings_name = row['Potential_matches_name']
  ratings_address = row['Potential_matches_address']
  ratings_businessid = row['Potential_matches_businessid']
  buiss_id = None
  if status == 'Active':
    if hospitality_address in ratings_address:
      counter = ratings_address.count(hospitality_address)
      if counter >1:
        all_scores = process.extract(hospitality_name, ratings_name, scorer=fuzz.token_set_ratio, limit=None)
        if all_scores[0][1]>=95:
          buiss_id = ratings_businessid[all_scores[0][2]]
      else:
        buiss_id = ratings_businessid[ratings_address.index(hospitality_address)]
  return buiss_id

In [ ]:
hospitality_df['Final_Business_ID'] = hospitality_df.apply(active_cases, axis=1)

In [ ]:
hospitality_clean = hospitality_df.dropna(subset = ['Final_Business_ID'])

In [ ]:
counts = hospitality_clean.groupby('Final_Business_ID')['Final_Business_ID'].transform('count')
df_filtered = hospitality_clean[counts > 1]

# 3. Create the final list
list_of_dfs = [group for x, group in df_filtered.groupby('Final_Business_ID')]

In [ ]:
list_of_dfs[0]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID
1218067,CORRIERI'S,SC861531,NaN,NaN,7 ALLOA ROAD,NaN,STIRLING,NaN,SCOTLAND,FK95LH,...,NaN,NaN,16/09/2026,NaN,56101,True,"[CAUSEWAYHEAD BOWLING CLUB, CORRIERI'S CAFE, L...","[ALLOA ROAD, 7 ALLOA ROAD, 17 ALLOA ROAD, 1 - ...","[1710, 1708, 9840, 1745727]",1708.0
4120544,R.A. CORRIERI & SONS,SC205129,NaN,NaN,7 ALLOA ROAD,NaN,STIRLING,NaN,NaN,FK95LH,...,NaN,NaN,30/03/2026,16/03/2025,56101,True,"[CAUSEWAYHEAD BOWLING CLUB, CORRIERI'S CAFE, L...","[ALLOA ROAD, 7 ALLOA ROAD, 17 ALLOA ROAD, 1 - ...","[1710, 1708, 9840, 1745727]",1708.0


In [ ]:
val = "SWEET PLANET CAFE"
search = ['LIDL UK LTD', 'PLANET SWEET', 'THE BASEMENT', "THE FORESTERS PUBLIC HOUSE ALSO T/A DISTRICT TACO & BURRITO BANDITS & DON TACO'TE"]
all_scores = process.extract(val, search, scorer=fuzz.token_set_ratio, limit=None)
print(all_scores)

[('PLANET SWEET', 100.0, 1), ('THE BASEMENT', 41.37931034482759, 2), ('LIDL UK LTD', 28.57142857142857, 0), ("THE FORESTERS PUBLIC HOUSE ALSO T/A DISTRICT TACO & BURRITO BANDITS & DON TACO'TE", 22.91666666666667, 3)]


In [ ]:
len(list_of_dfs)

3517

In [ ]:
val = "Dilkhush 1462 Pershore Road Birmingham B30 2NT England"
search = ['1462 Pershore Road']
all_scores = process.extract(val, search, scorer=fuzz.token_set_ratio, limit=None)
print(all_scores)

[('1462 Pershore Road', 100.0, 0)]


In [ ]:
def single_remove(row):
  name_list = row['Potential_matches_name']
  status = row['CompanyStatus']
  val = None
  if status == "Active":
    if len(name_list) == 1:
      val = True
    else:
      val = False
  return val

In [ ]:
hospitality_df['Single_remove'] = hospitality_df.apply(single_remove, axis=1)

In [ ]:
len(hospitality_df)

131032

In [ ]:
list_of_dfs[1]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID
114014,A.B.M.K,16990254,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,NaN,NaN,09/02/2027,NaN,56102,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",4833.0
4436579,SAUCY SAUSAGE CAFE,10227725,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,NaN,NaN,25/06/2026,11/06/2025,56103,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",4833.0


In [ ]:
def identify_actual_active(status, name_list, company_name, buissid_list):
  if status == "Active":
      all_scores = process.extract(company_name, name_list, scorer=fuzz.token_set_ratio, limit=None)
      if all_scores[0][1]>=90:
        return all_scores[0][1], buissid_list[all_scores[0][2]]
  return None, None

In [ ]:
def process_all_dfs(list_of_dfs):
    new_list = []
    for df in list_of_dfs:
        # Create a copy to avoid SettingWithCopyWarning
        df = df.copy()

        # Apply the function across columns (axis=1)
        # Result_type='expand' splits the returned tuple into two columns
        df[['MatchScore', 'Final_Business_ID']] = df.apply(
            lambda row: identify_actual_active(
                row['CompanyStatus'],
                row['Potential_matches_name'],      # Using the list in this row
                row['CompanyName'],
                row['Potential_matches_businessid']    # Using the list in this row
            ),
            axis=1,
            result_type='expand'
        )
        new_list.append(df)
    return new_list

# Apply the process
list_of_dfs = process_all_dfs(list_of_dfs)

In [ ]:
len(list_of_dfs)

3517

In [ ]:
list_of_dfs[1]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,MatchScore
114014,A.B.M.K,16990254,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,NaN,09/02/2027,NaN,56102,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",NaN,NaN
4436579,SAUCY SAUSAGE CAFE,10227725,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,NaN,25/06/2026,11/06/2025,56103,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",4833.0,100.0


In [ ]:
list_of_dfs[0]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,MatchScore
1218067,CORRIERI'S,SC861531,NaN,NaN,7 ALLOA ROAD,NaN,STIRLING,NaN,SCOTLAND,FK95LH,...,NaN,16/09/2026,NaN,56101,True,"[CAUSEWAYHEAD BOWLING CLUB, CORRIERI'S CAFE, L...","[ALLOA ROAD, 7 ALLOA ROAD, 17 ALLOA ROAD, 1 - ...","[1710, 1708, 9840, 1745727]",1708.0,100.0
4120544,R.A. CORRIERI & SONS,SC205129,NaN,NaN,7 ALLOA ROAD,NaN,STIRLING,NaN,NaN,FK95LH,...,NaN,30/03/2026,16/03/2025,56101,True,"[CAUSEWAYHEAD BOWLING CLUB, CORRIERI'S CAFE, L...","[ALLOA ROAD, 7 ALLOA ROAD, 17 ALLOA ROAD, 1 - ...","[1710, 1708, 9840, 1745727]",NaN,NaN


In [ ]:
active_duplicates = pd.concat(list_of_dfs)

/tmp/ipython-input-47541/1610502912.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  active_duplicates = pd.concat(list_of_dfs)


In [ ]:
active_duplicates.columns

Index(['CompanyName', ' CompanyNumber', 'RegAddress.CareOf',
       'RegAddress.POBox', 'RegAddress.AddressLine1',
       ' RegAddress.AddressLine2', 'RegAddress.PostTown', 'RegAddress.County',
       'RegAddress.Country', 'RegAddress.PostCode', 'CompanyCategory',
       'CompanyStatus', 'CountryOfOrigin', 'DissolutionDate',
       'IncorporationDate', 'Accounts.AccountRefDay',
       'Accounts.AccountRefMonth', 'Accounts.NextDueDate',
       'Accounts.LastMadeUpDate', 'Accounts.AccountCategory',
       'Returns.NextDueDate', 'Returns.LastMadeUpDate',
       'Mortgages.NumMortCharges', 'Mortgages.NumMortOutstanding',
       'Mortgages.NumMortPartSatisfied', 'Mortgages.NumMortSatisfied',
       'SICCode.SicText_1', 'SICCode.SicText_2', 'SICCode.SicText_3',
       'SICCode.SicText_4', 'LimitedPartnerships.NumGenPartners',
       'LimitedPartnerships.NumLimPartners', 'URI', 'PreviousName_1.CONDATE',
       ' PreviousName_1.CompanyName', ' PreviousName_2.CONDATE',
       ' PreviousName_2.C

In [ ]:
hospitality_df.head()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove
17,A TASTE OF TUSCANY,15761044,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,WC2H9JQ,...,NaN,17/06/2026,03/06/2025,56101,True,[MIA & BEN ORGANIC],[GARDEN STUDIOS],[1251615],NaN,True
46,COOL COFFEE LONDON,14428804,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,HA88BN,...,NaN,01/11/2026,18/10/2025,56102,True,"[ACTIHEALTH, ACTIKID, LITTLE LEADERS STONEGROV...","[COMMUNITY CENTRE 5 HAYLING WAY EDGWARE, COMMU...","[1722995, 1722996, 1087224, 1762881]",NaN,False
282,&FEAST,15236922,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,SW139HH,...,NaN,31/10/2026,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False
291,&IFOOD,12911325,NaN,NaN,316 GROVE GREEN ROAD,NaN,LONDON,NaN,ENGLAND,E114EA,...,NaN,11/10/2026,27/09/2025,56103,True,"[HEATHCOTE & STAR, SNS LOCAL, THE DRAGON FLAME...","[344 GROVE GREEN ROAD, GROUND FLOOR 320 GROVE ...","[1034984, 1888447, 1621047, 1073962]",1073962.0,False
297,&LKH,11943499,NaN,NaN,1 SOUTH PARADE,NaN,WESTON-SUPER-MARE,NORTH SOMERSET,UNITED KINGDOM,BS231JP,...,NaN,25/04/2026,11/04/2025,56302,True,[CORNISH CABIN (THE)],[ROYAL HOTEL],[1735331],NaN,True


In [ ]:
hospitality_df.set_index(' CompanyNumber', inplace=True)
active_duplicates.set_index(' CompanyNumber', inplace=True)

# 2. Add any columns from df_small that are missing in df_master
new_cols = active_duplicates.columns.difference(hospitality_df.columns)
for col in new_cols:
    hospitality_df[col] = None  # Initialize new columns with Nulls

# 3. Direct Assignment (Overwrites with values AND nulls)
hospitality_df.loc[active_duplicates.index, active_duplicates.columns] = active_duplicates

hospitality_df.reset_index(inplace=True)

In [ ]:
hospitality_df['IncorporationDate']

,IncorporationDate
0,2024-06-04
1,2022-10-19
2,2023-10-25
3,2020-09-28
4,2019-04-12
...,...
131027,2015-09-17
131028,2019-09-03
131029,2017-09-26
131030,2019-11-21


In [ ]:
hospitality_df.head()

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore
0,15761044,A TASTE OF TUSCANY,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,WC2H9JQ,...,17/06/2026,03/06/2025,56101,True,[MIA & BEN ORGANIC],[GARDEN STUDIOS],[1251615],NaN,True,None
1,14428804,COOL COFFEE LONDON,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,HA88BN,...,01/11/2026,18/10/2025,56102,True,"[ACTIHEALTH, ACTIKID, LITTLE LEADERS STONEGROV...","[COMMUNITY CENTRE 5 HAYLING WAY EDGWARE, COMMU...","[1722995, 1722996, 1087224, 1762881]",NaN,False,None
2,15236922,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,SW139HH,...,31/10/2026,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None
3,12911325,&IFOOD,NaN,NaN,316 GROVE GREEN ROAD,NaN,LONDON,NaN,ENGLAND,E114EA,...,11/10/2026,27/09/2025,56103,True,"[HEATHCOTE & STAR, SNS LOCAL, THE DRAGON FLAME...","[344 GROVE GREEN ROAD, GROUND FLOOR 320 GROVE ...","[1034984, 1888447, 1621047, 1073962]",NaN,False,NaN
4,11943499,&LKH,NaN,NaN,1 SOUTH PARADE,NaN,WESTON-SUPER-MARE,NORTH SOMERSET,UNITED KINGDOM,BS231JP,...,25/04/2026,11/04/2025,56302,True,[CORNISH CABIN (THE)],[ROYAL HOTEL],[1735331],NaN,True,None


In [ ]:
list_of_dfs[1]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,MatchScore
114014,A.B.M.K,16990254,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,NaN,09/02/2027,NaN,56102,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",NaN,NaN
4436579,SAUCY SAUSAGE CAFE,10227725,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,NaN,25/06/2026,11/06/2025,56103,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",4833.0,100.0


In [ ]:
len(hospitality_df)

131032

In [ ]:
len(hospitality_df)

131032

In [ ]:
len(list_of_dfs)

3517

In [ ]:
def identify_latest_active(df):
    # 1. Initialize the new column as False
    df['Is_Latest_Match'] = False

    # 2. Identify rows where MatchScore is NOT null
    # We only perform the "latest date" logic on these rows
    valid_mask = df['MatchScore'].notna()

    if valid_mask.any():
        # 3. Get the subset of valid rows
        valid_rows = df[valid_mask]

        # 4. Find the index of the row with the latest IncorporationDate
        # .idxmax() returns the index of the maximum value (latest date)
        latest_idx = valid_rows['IncorporationDate'].idxmax()

        # 5. Set that specific index to True
        df.at[latest_idx, 'Is_Latest_Match'] = True

    return df

# Apply the function to your list of DataFrames
list_of_dfs = [identify_latest_active(d.copy()) for d in list_of_dfs]

In [ ]:
list_of_dfs[10]

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,MatchScore,Is_Latest_Match
545644,AZZOU ENTERPRISE,12351585,NaN,NaN,25 STATION ROAD,NaN,HORLEY,NaN,ENGLAND,RH69HW,...,19/12/2024,05/12/2023,56101,True,"[BLUE LAKE RESTAURANT, THE CURRY BENGAL]","[11-13 STATION ROAD, 25 STATION ROAD]","[49560, 14797]",None,None,False
1368119,DAWWAT,14697943,NaN,NaN,25 STATION ROAD,NaN,HORLEY,NaN,ENGLAND,RH69HW,...,14/03/2026,28/02/2025,56101,True,"[BLUE LAKE RESTAURANT, THE CURRY BENGAL]","[11-13 STATION ROAD, 25 STATION ROAD]","[49560, 14797]",None,None,False


In [ ]:
active_duplicates = pd.concat(list_of_dfs)

/tmp/ipython-input-47541/1610502912.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  active_duplicates = pd.concat(list_of_dfs)


In [ ]:
active_duplicates.head()

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,MatchScore,Is_Latest_Match
1218067,CORRIERI'S,SC861531,NaN,NaN,7 ALLOA ROAD,NaN,STIRLING,NaN,SCOTLAND,FK95LH,...,16/09/2026,NaN,56101,True,"[CAUSEWAYHEAD BOWLING CLUB, CORRIERI'S CAFE, L...","[ALLOA ROAD, 7 ALLOA ROAD, 17 ALLOA ROAD, 1 - ...","[1710, 1708, 9840, 1745727]",1708.0,100.0,True
4120544,R.A. CORRIERI & SONS,SC205129,NaN,NaN,7 ALLOA ROAD,NaN,STIRLING,NaN,NaN,FK95LH,...,30/03/2026,16/03/2025,56101,True,"[CAUSEWAYHEAD BOWLING CLUB, CORRIERI'S CAFE, L...","[ALLOA ROAD, 7 ALLOA ROAD, 17 ALLOA ROAD, 1 - ...","[1710, 1708, 9840, 1745727]",NaN,NaN,False
114014,A.B.M.K,16990254,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,09/02/2027,NaN,56102,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",NaN,NaN,False
4436579,SAUCY SAUSAGE CAFE,10227725,NaN,NaN,259 LOWER ADDISCOMBE ROAD,NaN,CROYDON,NaN,ENGLAND,CR06RD,...,25/06/2026,11/06/2025,56103,True,"[DAY 1 EXPRESS, FRESHCO, GREGGS, KFC, PIZZA ON...","[GROUND FLOOR SHOP, 243 - 245 LOWER ADDISCOMBE...","[1815623, 1799488, 4832, 859725, 10592, 4833, ...",4833.0,100.0,True
4063125,PTN THAI CATERING,16857008,NaN,NaN,2 ORTON BUILDINGS,PORTLAND ROAD,LONDON,NaN,ENGLAND,SE254UD,...,27/11/2026,NaN,56101,True,"[LUSITANO CAFE AND DELI, MANTANAH THAI CUISINE...","[3 ORTON BUILDINGS, 2 ORTON BUILDINGS, 1 ORTON...","[1025012, 5308, 5307]",NaN,NaN,False


In [ ]:
active_duplicates.to_excel('Active Duplicates.xlsx',index = False)

In [ ]:
hospitality_df.columns

Index([' CompanyNumber', 'CompanyName', 'RegAddress.CareOf',
       'RegAddress.POBox', 'RegAddress.AddressLine1',
       ' RegAddress.AddressLine2', 'RegAddress.PostTown', 'RegAddress.County',
       'RegAddress.Country', 'RegAddress.PostCode', 'CompanyCategory',
       'CompanyStatus', 'CountryOfOrigin', 'DissolutionDate',
       'IncorporationDate', 'Accounts.AccountRefDay',
       'Accounts.AccountRefMonth', 'Accounts.NextDueDate',
       'Accounts.LastMadeUpDate', 'Accounts.AccountCategory',
       'Returns.NextDueDate', 'Returns.LastMadeUpDate',
       'Mortgages.NumMortCharges', 'Mortgages.NumMortOutstanding',
       'Mortgages.NumMortPartSatisfied', 'Mortgages.NumMortSatisfied',
       'SICCode.SicText_1', 'SICCode.SicText_2', 'SICCode.SicText_3',
       'SICCode.SicText_4', 'LimitedPartnerships.NumGenPartners',
       'LimitedPartnerships.NumLimPartners', 'URI', 'PreviousName_1.CONDATE',
       ' PreviousName_1.CompanyName', ' PreviousName_2.CONDATE',
       ' PreviousName_2.C

In [ ]:
hospitality_df.reset_index(inplace=True)
hospitality_df.head()

,index,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,...,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore
0,0,15761044,A TASTE OF TUSCANY,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,...,17/06/2026,03/06/2025,56101,True,[MIA & BEN ORGANIC],[GARDEN STUDIOS],[1251615],NaN,True,None
1,1,14428804,COOL COFFEE LONDON,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,...,01/11/2026,18/10/2025,56102,True,"[ACTIHEALTH, ACTIKID, LITTLE LEADERS STONEGROV...","[COMMUNITY CENTRE 5 HAYLING WAY EDGWARE, COMMU...","[1722995, 1722996, 1087224, 1762881]",NaN,False,None
2,2,15236922,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,...,31/10/2026,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None
3,3,12911325,&IFOOD,NaN,NaN,316 GROVE GREEN ROAD,NaN,LONDON,NaN,ENGLAND,...,11/10/2026,27/09/2025,56103,True,"[HEATHCOTE & STAR, SNS LOCAL, THE DRAGON FLAME...","[344 GROVE GREEN ROAD, GROUND FLOOR 320 GROVE ...","[1034984, 1888447, 1621047, 1073962]",NaN,False,NaN
4,4,11943499,&LKH,NaN,NaN,1 SOUTH PARADE,NaN,WESTON-SUPER-MARE,NORTH SOMERSET,UNITED KINGDOM,...,25/04/2026,11/04/2025,56302,True,[CORNISH CABIN (THE)],[ROYAL HOTEL],[1735331],NaN,True,None


In [ ]:
hospitality_df.set_index(' CompanyNumber', inplace=True)
active_duplicates.set_index(' CompanyNumber', inplace=True)

# 2. Add any columns from df_small that are missing in df_master
new_cols = active_duplicates.columns.difference(hospitality_df.columns)
for col in new_cols:
    hospitality_df[col] = None  # Initialize new columns with Nulls

# 3. Direct Assignment (Overwrites with values AND nulls)
hospitality_df.loc[active_duplicates.index, active_duplicates.columns] = active_duplicates

hospitality_df.reset_index(inplace=True)

In [ ]:
len(hospitality_df)

131032

In [ ]:
hospitality_df.head()

,CompanyNumber,index,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,...,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match
0,15761044,0,A TASTE OF TUSCANY,NaN,NaN,71-75 SHELTON STREET,COVENT GARDEN,LONDON,NaN,UNITED KINGDOM,...,03/06/2025,56101,True,[MIA & BEN ORGANIC],[GARDEN STUDIOS],[1251615],NaN,True,None,None
1,14428804,1,COOL COFFEE LONDON,NaN,NaN,FLAT 9 METROPOLITAN COURT,6 HAYLING WAY,EDGWARE,NaN,ENGLAND,...,18/10/2025,56102,True,"[ACTIHEALTH, ACTIKID, LITTLE LEADERS STONEGROV...","[COMMUNITY CENTRE 5 HAYLING WAY EDGWARE, COMMU...","[1722995, 1722996, 1087224, 1762881]",NaN,False,None,None
2,15236922,2,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,...,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None,None
3,12911325,3,&IFOOD,NaN,NaN,316 GROVE GREEN ROAD,NaN,LONDON,NaN,ENGLAND,...,27/09/2025,56103,True,"[HEATHCOTE & STAR, SNS LOCAL, THE DRAGON FLAME...","[344 GROVE GREEN ROAD, GROUND FLOOR 320 GROVE ...","[1034984, 1888447, 1621047, 1073962]",NaN,False,NaN,False
4,11943499,4,&LKH,NaN,NaN,1 SOUTH PARADE,NaN,WESTON-SUPER-MARE,NORTH SOMERSET,UNITED KINGDOM,...,11/04/2025,56302,True,[CORNISH CABIN (THE)],[ROYAL HOTEL],[1735331],NaN,True,None,None


In [ ]:
hospitality_df.isna().sum()

,0
CompanyNumber,0
index,0
CompanyName,0
RegAddress.CareOf,130657
RegAddress.POBox,130971
...,...
Potential_matches_businessid,0
Final_Business_ID,106562
Single_remove,22628
MatchScore,129212


In [ ]:
# Identify rows that are duplicates within the 'True' group (excluding the first)
mask = (hospitality_df['Single_remove'] == True) & hospitality_df.duplicated(subset='Final_Business_ID', keep='first')

# Keep everything EXCEPT the rows identified by that mask
hospitality_df = hospitality_df[~mask]

In [ ]:
bad_rows_mask = (hospitality_df['CompanyStatus'] == 'Active') & (hospitality_df['Final_Business_ID'].isna())

# 2. Keep everything that is NOT a bad row
hospitality_df = hospitality_df[~bad_rows_mask]

In [ ]:
len(hospitality_df)

47048

In [ ]:
hospitality_df.head()

,CompanyNumber,index,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,...,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match
2,15236922,2,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,...,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None,None
5,16212604,5,&YAT SING,NaN,NaN,316 GROVE GREEN ROAD,NaN,LONDON,NaN,UNITED KINGDOM,...,NaN,56103,True,"[HEATHCOTE & STAR, SNS LOCAL, THE DRAGON FLAME...","[344 GROVE GREEN ROAD, GROUND FLOOR 320 GROVE ...","[1034984, 1888447, 1621047, 1073962]",1073962.0,False,94.117647,False
12,16731283,12,CITY PIZZA LEEDS,NaN,NaN,5 EASY ROAD,NaN,LEEDS,NaN,ENGLAND,...,NaN,56103,True,"[CITY PIZZA, THE NEW CAPTAINS TABLE]","[5 EASY ROAD, 1 EASY ROAD]","[1894924, 855837]",1894924.0,False,100.0,True
13,15466179,13,-THE MARULA TREE,NaN,NaN,TAMAR HOUSE,ANTONY ROAD,TORPOINT,NaN,ENGLAND,...,04/02/2025,56101,True,"[CHEF DEBBIE THORPE, THE MARULA TREE]","[TAMAR HOUSE, TAMAR HOUSE]","[1722858, 1722870]",1722870.0,False,None,None
17,11681186,17,01 YENI,NaN,NaN,485 KINGSLAND ROAD,NaN,LONDON,NaN,UNITED KINGDOM,...,15/11/2020,56102,True,[TAMARIS FRENCH TACOS],[485 KINGSLAND ROAD],[1751237],NaN,None,None,None


In [ ]:
hospitality_df = hospitality_df.drop(['index'],axis = 1)

In [ ]:
hospitality_df = hospitality_df[hospitality_df['Is_Latest_Match']!=False]

In [ ]:
len(hospitality_df)

46668

In [ ]:
hospitality_df['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,24040
Active - Proposal to Strike off,18111
Liquidation,4405
In Administration,74
Voluntary Arrangement,19
In Administration/Administrative Receiver,10
Live but Receiver Manager on at least one charge,9


In [ ]:
hospitality_df.head()

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match
2,15236922,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,SW139HH,...,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None,None
12,16731283,CITY PIZZA LEEDS,NaN,NaN,5 EASY ROAD,NaN,LEEDS,NaN,ENGLAND,LS98LH,...,NaN,56103,True,"[CITY PIZZA, THE NEW CAPTAINS TABLE]","[5 EASY ROAD, 1 EASY ROAD]","[1894924, 855837]",1894924.0,False,100.0,True
13,15466179,-THE MARULA TREE,NaN,NaN,TAMAR HOUSE,ANTONY ROAD,TORPOINT,NaN,ENGLAND,PL112JS,...,04/02/2025,56101,True,"[CHEF DEBBIE THORPE, THE MARULA TREE]","[TAMAR HOUSE, TAMAR HOUSE]","[1722858, 1722870]",1722870.0,False,None,None
17,11681186,01 YENI,NaN,NaN,485 KINGSLAND ROAD,NaN,LONDON,NaN,UNITED KINGDOM,E84AU,...,15/11/2020,56102,True,[TAMARIS FRENCH TACOS],[485 KINGSLAND ROAD],[1751237],NaN,None,None,None
19,10393013,036,NaN,NaN,47 CRICKLEWOOD BROADWAY,NaN,LONDON,NaN,NaN,NW23JX,...,25/09/2021,56101,True,"[ABYSSINIA RESTAURANT AND BAR, AL QAMAR GRILL,...","[9 CRICKLEWOOD BROADWAY, 33 CRICKLEWOOD BROADW...","[533466, 533469, 1783837, 1418851, 1515962, 53...",NaN,None,None,None


In [ ]:
hospitality_df.to_excel('Active Deleted.xlsx',index = False)

In [ ]:
print(hospitality_df.columns)

Index([' CompanyNumber', 'CompanyName', 'RegAddress.CareOf',
       'RegAddress.POBox', 'RegAddress.AddressLine1',
       ' RegAddress.AddressLine2', 'RegAddress.PostTown', 'RegAddress.County',
       'RegAddress.Country', 'RegAddress.PostCode', 'CompanyCategory',
       'CompanyStatus', 'CountryOfOrigin', 'DissolutionDate',
       'IncorporationDate', 'Accounts.AccountRefDay',
       'Accounts.AccountRefMonth', 'Accounts.NextDueDate',
       'Accounts.LastMadeUpDate', 'Accounts.AccountCategory',
       'Returns.NextDueDate', 'Returns.LastMadeUpDate',
       'Mortgages.NumMortCharges', 'Mortgages.NumMortOutstanding',
       'Mortgages.NumMortPartSatisfied', 'Mortgages.NumMortSatisfied',
       'SICCode.SicText_1', 'SICCode.SicText_2', 'SICCode.SicText_3',
       'SICCode.SicText_4', 'LimitedPartnerships.NumGenPartners',
       'LimitedPartnerships.NumLimPartners', 'URI', 'PreviousName_1.CONDATE',
       ' PreviousName_1.CompanyName', ' PreviousName_2.CONDATE',
       ' PreviousName_2.C

In [ ]:
ratings_df.head()

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessTypeID,FHRSID,BusinessName,BusinessType,ConfidenceInManagement,Hygiene,...,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural,PostCode_Validity
0,72 REGENT STREET,NaN,Cambridge,Cambridgeshire,1,1472964,@72.CHINA,Restaurant/Cafe/Canteen,5.0,10.0,...,0.126139,False,CB21DP,2024-05-14,fhrs_4_en-GB,4,NaN,FHRS,5.0,True
1,84 REGENT STREET,NaN,Cambridge,Cambridgeshire,1,1472953,1 AND 1 ROUGAMO LTD,Restaurant/Cafe/Canteen,10.0,10.0,...,0.126407,False,CB21DP,2024-04-17,fhrs_3_en-GB,3,NaN,FHRS,10.0,True
2,108 CHERRY HINTON ROAD,NaN,Cambridge,Cambridgeshire,4613,1888757,108 CHERRY HINTON MINI MARKET,Retailers - other,0.0,5.0,...,0.141757,False,CB17AJ,2024-12-17,fhrs_4_en-GB,4,NaN,FHRS,10.0,True
3,2 KING STREET,NaN,Cambridge,Cambridgeshire,1,1908335,13 WEEKS,Restaurant/Cafe/Canteen,NaN,NaN,...,0.121618,False,CB11LN,NaN,fhrs_awaitinginspection_en-GB,AwaitingInspection,NaN,FHRS,NaN,True
4,196 MILL ROAD,NaN,Cambridge,Cambridgeshire,1,1472773,196,Restaurant/Cafe/Canteen,0.0,5.0,...,0.144063,False,CB13NF,2026-02-25,fhrs_5_en-GB,5,NaN,FHRS,5.0,True


In [ ]:
def extract_uk_postcode(address):
    if not isinstance(address, str):
        return None
    pattern = r'([A-Z][A-HJ-Y]?[0-9][A-Z0-9]? ?[0-9][A-Z]{2}|GIR ?0AA)'
    match = re.search(pattern, address.upper())
    if match:
        candidate = match.group(0)
        if checkpost(candidate):
            return candidate
    return None

In [ ]:
extract_uk_postcode("1 Belgrave Middleway")

In [ ]:
def separate_num_alpha(text):
  if not isinstance(text, str) or not text.strip():
    return [None, None, None]
  postcode = extract_uk_postcode(text)
  if postcode:
    text = text.replace(postcode, '').strip()
    formatted_postcode = postcode.replace(' ', '')
  else:
    formatted_postcode = None
  parts = text.split()

    # Index 0: Parts with at least one digit
  with_numbers = " ".join([p for p in parts if re.search(r'\d', p)])

    # Index 1: Parts with NO digits
  no_numbers = " ".join([p for p in parts if not re.search(r'\d', p)])

  return [with_numbers, no_numbers, formatted_postcode]

In [ ]:
text = "1 Belgrave Middleway"

In [ ]:
separate_num_alpha(text)

['1', 'Belgrave Middleway', None]

In [ ]:
from tqdm import tqdm
tqdm.pandas()

In [ ]:
import time

In [ ]:
hospitality_df.head()

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,ConfStmtLastMadeUpDate,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match
2,15236922,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,SW139HH,...,17/10/2025,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None,None
12,16731283,CITY PIZZA LEEDS,NaN,NaN,5 EASY ROAD,NaN,LEEDS,NaN,ENGLAND,LS98LH,...,NaN,56103,True,"[CITY PIZZA, THE NEW CAPTAINS TABLE]","[5 EASY ROAD, 1 EASY ROAD]","[1894924, 855837]",1894924.0,False,100.0,True
13,15466179,-THE MARULA TREE,NaN,NaN,TAMAR HOUSE,ANTONY ROAD,TORPOINT,NaN,ENGLAND,PL112JS,...,04/02/2025,56101,True,"[CHEF DEBBIE THORPE, THE MARULA TREE]","[TAMAR HOUSE, TAMAR HOUSE]","[1722858, 1722870]",1722870.0,False,None,None
17,11681186,01 YENI,NaN,NaN,485 KINGSLAND ROAD,NaN,LONDON,NaN,UNITED KINGDOM,E84AU,...,15/11/2020,56102,True,[TAMARIS FRENCH TACOS],[485 KINGSLAND ROAD],[1751237],NaN,None,None,None
19,10393013,036,NaN,NaN,47 CRICKLEWOOD BROADWAY,NaN,LONDON,NaN,NaN,NW23JX,...,25/09/2021,56101,True,"[ABYSSINIA RESTAURANT AND BAR, AL QAMAR GRILL,...","[9 CRICKLEWOOD BROADWAY, 33 CRICKLEWOOD BROADW...","[533466, 533469, 1783837, 1418851, 1515962, 53...",NaN,None,None,None


In [ ]:
hospitality_df['Potential_matches_address'].isna().sum()

np.int64(0)

In [ ]:
def non_active(ratings_df, hospitality_df_row):
    if hospitality_df_row['CompanyStatus'] != "Active":
        company_name = hospitality_df_row['CompanyName']
        name_list = hospitality_df_row['Potential_matches_name']
        businessid_list = hospitality_df_row['Potential_matches_businessid']

        # 1. Initial Name Match Check
        name_scores = process.extract(company_name, name_list, scorer=fuzz.token_set_ratio, limit=None)
        if name_scores and name_scores[0][1] >= 90:
            return (businessid_list[name_scores[0][2]], "Direct_Name", hospitality_df_row)

        # 2. API CALL
        company_number = str(hospitality_df_row[' CompanyNumber']).strip() # Remove accidental spaces
        url = f"https://api.company-information.service.gov.uk/company/{company_number}/filing-history"

        try:
            response = requests.get(url, auth=(API_KEY, ''), timeout=10)
            if response.status_code == 200:
                time.sleep(0.5)
                data = response.json()
                filings = data.get('items', [])
                df = pd.DataFrame(filings)

                ad01_indices = df.index[df['type'] == 'AD01']
                if not ad01_indices.empty:
                    last_idx = ad01_indices[-1]
                    hist_address = df.loc[last_idx, 'description_values'].get('old_address')

                    if hist_address:
                        given_address_split = separate_num_alpha(hist_address)
                        postcode = given_address_split[2]

                        # --- THE NON-NEGOTIABLE OVERWRITE ---
                        # We save these into the row Series so they persist back to the main DF
                        hospitality_df_row['RegAddress.AddressLine1'] = hist_address
                        hospitality_df_row['RegAddress.PostCode'] = postcode

                        if postcode is not None:
                            mask = ratings_df['PostCode'] == postcode
                            h_ids = ratings_df.loc[mask, 'FHRSID'].tolist()
                            h_names = ratings_df.loc[mask, 'BusinessName'].tolist()
                            h_addrs = ratings_df.loc[mask, 'AddressLine1'].tolist()

                            # Overwrite potential candidate lists
                            hospitality_df_row['Potential_matches_name'] = h_names
                            hospitality_df_row['Potential_matches_businessid'] = h_ids
                            hospitality_df_row['Potential_matches_address'] = h_addrs

                            # Historical Name Match
                            n_scores = process.extract(company_name, h_names, scorer=fuzz.token_set_ratio)
                            if n_scores and n_scores[0][1] >= 90:
                                return (h_ids[n_scores[0][2]], "API_Name", hospitality_df_row)

                            # Historical Address Match
                            address_split_list = [separate_num_alpha(x) for x in h_addrs]
                            for x, i in enumerate(address_split_list):
                                a_scores = process.extract(given_address_split[1], [i[1]], scorer=fuzz.token_set_ratio)
                                if a_scores and a_scores[0][1] >= 90:
                                    if given_address_split[0] == i[0]:
                                        return (h_ids[x], "API_Address", hospitality_df_row)

            return (None, "API_No_Match_Found", hospitality_df_row)
        except Exception as e:
            return (None, f"API_Error: {str(e)}", hospitality_df_row)

    return (None, "Active_Skipped", hospitality_df_row)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# 1. Filter for Non-Active rows to optimize time and API calls
non_active_mask = hospitality_df['CompanyStatus'] != "Active"
indices_to_process = hospitality_df[non_active_mask].index

results_dict = {}

# 2. Parallel Processing with Progress Bar
# max_workers=10 is safe for Companies House rate limits (600/5min)
with ThreadPoolExecutor(max_workers=10) as executor:
    # We pass .copy() to ensure thread isolation
    future_to_idx = {
        executor.submit(non_active, ratings_df, hospitality_df.loc[idx].copy()): idx
        for idx in indices_to_process
    }

    for future in tqdm(as_completed(future_to_idx), total=len(indices_to_process), desc="Matching"):
        results_dict[future_to_idx[future]] = future.result()

# 3. THE NON-NEGOTIABLE OVERWRITE
# We extract the modified data from results_dict and write it into the main DF
print("Applying overwrites to main DataFrame...")
for idx, result in results_dict.items():
    if result:
        match_id, method, updated_row = result
        # Permanently overwrite the row with the recovered Address, Postcode, and Lists
        hospitality_df.loc[idx] = updated_row
        # Set the final results
        hospitality_df.at[idx, 'Final_Business_ID'] = match_id
        hospitality_df.at[idx, 'Match_Method'] = method

print("Done! Check your hospitality_df for updated values.")

Matching: 100%|██████████| 22628/22628 [14:54<00:00, 25.31it/s]


Applying overwrites to main DataFrame...
Done! Check your hospitality_df for updated values.


In [ ]:
hospitality_df['Final_Business_ID'].isna().sum()

np.int64(20841)

In [ ]:
len(hospitality_df)

46668

In [ ]:
hospitality_df.head()

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match,Match_Method
2,15236922,&FEAST,NaN,NaN,57 CHURCH ROAD,NaN,LONDON,NaN,ENGLAND,SW139HH,...,56101,True,"[& FEAST, BARNES PANTRY, GAIL'S BARNES, NATSON...","[57 CHURCH ROAD, 63-65 CHURCH ROAD, 49 CHURCH ...","[1574552, 1654955, 1574324, 1574375, 1574551, ...",1574552.0,False,None,None,NaN
12,16731283,CITY PIZZA LEEDS,NaN,NaN,5 EASY ROAD,NaN,LEEDS,NaN,ENGLAND,LS98LH,...,56103,True,"[CITY PIZZA, THE NEW CAPTAINS TABLE]","[5 EASY ROAD, 1 EASY ROAD]","[1894924, 855837]",1894924.0,False,100.0,True,NaN
13,15466179,-THE MARULA TREE,NaN,NaN,TAMAR HOUSE,ANTONY ROAD,TORPOINT,NaN,ENGLAND,PL112JS,...,56101,True,"[CHEF DEBBIE THORPE, THE MARULA TREE]","[TAMAR HOUSE, TAMAR HOUSE]","[1722858, 1722870]",1722870.0,False,None,None,NaN
17,11681186,01 YENI,NaN,NaN,485 KINGSLAND ROAD,NaN,LONDON,NaN,UNITED KINGDOM,E84AU,...,56102,True,[TAMARIS FRENCH TACOS],[485 KINGSLAND ROAD],[1751237],NaN,None,None,None,API_No_Match_Found
19,10393013,036,NaN,NaN,PO Box 4385 10393013: Companies House Default ...,NaN,LONDON,NaN,NaN,CF148LH,...,56101,True,[],[],[],NaN,None,None,None,API_No_Match_Found


In [ ]:
hospitality_df['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,24040
Active - Proposal to Strike off,18111
Liquidation,4405
In Administration,74
Voluntary Arrangement,19
In Administration/Administrative Receiver,10
Live but Receiver Manager on at least one charge,9


In [ ]:
len(hospitality_df[hospitality_df['CompanyStatus']!='Active'])

22628

In [ ]:
hospitality_df[hospitality_df['CompanyStatus']!='Active']['Final_Business_ID'].isna().sum()

np.int64(20841)

In [ ]:
hospitality_df.to_excel('Check All.xlsx',index = False)

In [ ]:
ratings_df[ratings_df['FHRSID']==465257]

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessTypeID,FHRSID,BusinessName,BusinessType,ConfidenceInManagement,Hygiene,...,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural,PostCode_Validity
177586,78 HIGH STREET,Gosforth,Newcastle Upon Tyne,NaN,1,465257,AHAD TANDOORI,Restaurant/Cafe/Canteen,NaN,NaN,...,-1.619739,False,NE31HB,2026-01-26,fhrs_4_en-GB,4,NaN,FHRS,NaN,True


In [ ]:
ratings_df[ratings_df['FHRSID']==465257]

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessTypeID,FHRSID,BusinessName,BusinessType,ConfidenceInManagement,Hygiene,...,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural,PostCode_Validity
177586,78 HIGH STREET,Gosforth,Newcastle Upon Tyne,NaN,1,465257,AHAD TANDOORI,Restaurant/Cafe/Canteen,NaN,NaN,...,-1.619739,False,NE31HB,2026-01-26,fhrs_4_en-GB,4,NaN,FHRS,NaN,True


In [ ]:
company_number = "11555350"  # Example company
url = f"https://api.company-information.service.gov.uk/company/{company_number}/filing-history"
response = requests.get(url, auth=(API_KEY, ''))

if response.status_code == 200:
    data = response.json()
    #print(data)
    filings = data.get('items', [])
    # # Create the DataFrame
    df = pd.DataFrame(filings)

    # # Display the first few rows
    # print(f"Successfully retrieved {len(df)} filings.")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

In [ ]:
df

,transaction_id,barcode,type,date,category,subcategory,description,description_values,pages,action_date,paper_filed,links,resolutions,associated_filings
0,MzQ3MjM3ODAwNWFkaXF6a2N4,YE5XXYJM,LIQ03,2025-07-08,insolvency,voluntary,liquidation-voluntary-statement-of-receipts-an...,{'brought_down_date': '2025-05-08'},12,2025-05-08,True,{'self': '/company/11555350/filing-history/MzQ...,NaN,NaN
1,MzQyMTM2ODY5NWFkaXF6a2N4,YD315BJ4,LIQ02,2024-05-19,insolvency,voluntary,liquidation-voluntary-statement-of-affairs,NaN,10,NaN,True,{'self': '/company/11555350/filing-history/MzQ...,NaN,NaN
2,MzQyMTM2ODU1N2FkaXF6a2N4,YD315B16,600,2024-05-19,insolvency,voluntary,liquidation-voluntary-appointment-of-liquidator,NaN,3,NaN,True,{'self': '/company/11555350/filing-history/MzQ...,NaN,NaN
3,WUQzMTVDU1phZGlxemtjeA,NaN,RESOLUTIONS,2024-05-19,resolution,NaN,resolution,{'description': 'Resolutions'},1,NaN,True,{'self': '/company/11555350/filing-history/WUQ...,"[{'category': 'liquidation', 'description': 'l...",NaN
4,MzQxOTA1MDQxMmFkaXF6a2N4,YD1E2GYA,AD01,2024-04-29,address,NaN,change-registered-office-address-company-with-...,"{'change_date': '2024-04-29', 'old_address': '...",3,2024-04-29,True,{'self': '/company/11555350/filing-history/MzQ...,NaN,NaN
5,MzM5MjM3ODY5OWFkaXF6a2N4,NaN,DISS16(SOAS),2023-09-09,dissolution,NaN,dissolved-compulsory-strike-off-suspended,NaN,1,NaN,True,{'self': '/company/11555350/filing-history/MzM...,NaN,NaN
6,MzM5MDE3MTg0N2FkaXF6a2N4,NaN,GAZ1,2023-08-29,gazette,NaN,gazette-notice-compulsory,NaN,1,NaN,True,{'self': '/company/11555350/filing-history/MzM...,NaN,NaN
7,MzM1MTIyOTY3MWFkaXF6a2N4,XBC33VRG,CS01,2022-09-08,confirmation-statement,NaN,confirmation-statement-with-no-updates,{'made_up_date': '2022-09-02'},3,2022-09-02,NaN,{'self': '/company/11555350/filing-history/MzM...,NaN,NaN
8,MzM0MzAzNDMxNmFkaXF6a2N4,XB6KSL8B,AA,2022-06-20,accounts,NaN,accounts-with-accounts-type-dormant,{'made_up_date': '2021-09-30'},2,2021-09-30,NaN,{'self': '/company/11555350/filing-history/MzM...,NaN,NaN
9,MzMxMzc3NTg5NmFkaXF6a2N4,XACZXHS1,CS01,2021-09-14,confirmation-statement,NaN,confirmation-statement-with-no-updates,{'made_up_date': '2021-09-02'},3,2021-09-02,NaN,{'self': '/company/11555350/filing-history/MzM...,NaN,NaN


In [ ]:
hospitality_df = hospitality_df.dropna(subset=['Final_Business_ID'])

In [ ]:
len(hospitality_df)

25827

In [ ]:
hospitality_df['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,24040
Active - Proposal to Strike off,1745
Liquidation,40
Voluntary Arrangement,2


In [ ]:
ratings_df[ratings_df['FHRSID']==1903046]

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessTypeID,FHRSID,BusinessName,BusinessType,ConfidenceInManagement,Hygiene,...,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural,PostCode_Validity
268458,8-12 BRADWALL ROAD,Sandbach,NaN,Cheshire,1,1903046,GANGES,Restaurant/Cafe/Canteen,NaN,NaN,...,-2.365291,False,CW111GB,2025-10-08,fhrs_5_en-GB,5,NaN,FHRS,NaN,True


In [ ]:
hospitality_df['Match_Method'].value_counts()

,count
Match_Method,
Direct_Name,1780
API_Name,7


In [ ]:
# 1. Ensure IncorporationDate is in datetime format for correct sorting
hospitality_df['IncorporationDate'] = pd.to_datetime(hospitality_df['IncorporationDate'])

# 2. Sort by Final_Business_ID and IncorporationDate (latest date at the top)
# We put NaT (missing dates) at the end
hospitality_df = hospitality_df.sort_values(
    by=['Final_Business_ID', 'IncorporationDate'],
    ascending=[True, False],
    na_position='last'
)

# 3. Remove duplicates based on Final_Business_ID
# keep='first' works here because we sorted the latest dates to the top
# We exclude rows where Final_Business_ID is None/NaN so we don't lose unmatched data
matched_mask = hospitality_df['Final_Business_ID'].notna()

hospitality_df_cleaned = pd.concat([
    hospitality_df[matched_mask].drop_duplicates(subset=['Final_Business_ID'], keep='first'),
    hospitality_df[~matched_mask]
])

print(f"Original rows: {len(hospitality_df)}")
print(f"Rows after removing duplicate matches: {len(hospitality_df_cleaned)}")

Original rows: 25827
Rows after removing duplicate matches: 25268


In [ ]:
hospitality_df_cleaned['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,24002
Active - Proposal to Strike off,1235
Liquidation,29
Voluntary Arrangement,2


In [ ]:
(1235+29+2)/(1235+29+2+24002)

0.050102896944752257

In [ ]:
counts = hospitality_df.groupby('Final_Business_ID')['Final_Business_ID'].transform('count')
df_filtered = hospitality_df[counts > 1]

# 3. Create the final list
list_of_dfs = [group for x, group in df_filtered.groupby('Final_Business_ID')]

In [ ]:
list_of_dfs[1]

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match,Match_Method
51791,16790783,HONGS PLACE FOOD,NaN,NaN,86 FOREST DRIVE,NaN,CHELMSFORD,NaN,ENGLAND,CM12TS,...,56103,True,"[CHURCH OF THE HOLY SPIRIT, FOREST DRIVE POST ...","[FOREST DRIVE, 56 FOREST DRIVE, 86 FOREST DRIV...","[24021, 1751702, 14864, 1808639, 25923, 24020,...",14864.0,False,100.0,True,NaN
51792,14455419,HONGS PLACE,NaN,NaN,86 FOREST DRIVE,NaN,CHELMSFORD,NaN,ENGLAND,CM12TS,...,56103,True,"[CHURCH OF THE HOLY SPIRIT, FOREST DRIVE POST ...","[FOREST DRIVE, 56 FOREST DRIVE, 86 FOREST DRIV...","[24021, 1751702, 14864, 1808639, 25923, 24020,...",14864.0,None,None,None,Direct_Name


In [ ]:
patch_data = {}

for sub_df in list_of_dfs:
    if sub_df.empty:
        continue
    sub_df['IncorporationDate'] = pd.to_datetime(sub_df['IncorporationDate'], errors='coerce')
    sub_df = sub_df.sort_values('IncorporationDate', ascending=False)
    for i, idx in enumerate(sub_df.index):
        patch_data[idx] = (i == 0) # True for the first, False for the rest
patch_series = pd.Series(patch_data, name='Is_Latest_Match')
hospitality_df.update(patch_series.to_frame())

print(f"Surgical update complete. {len(patch_series)} specific rows updated.")

Surgical update complete. 1109 specific rows updated.


In [ ]:
hospitality_df.head(10)

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match,Match_Method
68397,SC780693,M&S MAZZONI,NaN,NaN,6 BAY STREET,NaN,PORT GLASGOW,NaN,SCOTLAND,PA145ED,...,56103,True,[LOUIS'S],[6 BAY STREET],[40],40.0,True,None,None,NaN
65403,SC402065,LIGHTHOUSE BAR,NaN,NaN,37 CATHCART STREET,NaN,GREENOCK,RENFREWSHIRE,SCOTLAND,PA151DG,...,56302,True,"[CHINA GARDEN, FISHERMANS DOCK, LIGHTHOUSE BAR]","[47 - 49 CATHCART STREET, 43 CATHCART STREET, ...","[1551791, 1786883, 113]",113.0,False,None,None,NaN
93534,SC444940,RISTORANTE ARLECCHINO,NaN,NaN,68 WEST BLACKHALL STREET,NaN,GREENOCK,NaN,SCOTLAND,PA151XG,...,56101,True,[L'ARLECCHINO RESTAURANT],[68 WEST BLACKHALL STREET],[299],299.0,True,None,None,NaN
3003,SC857241,AF EATS,NaN,NaN,78 INVERKIP ROAD,NaN,GREENOCK,NaN,SCOTLAND,PA169AP,...,56103,True,"[BARRS COTTAGE CHIP SHOP, MORRISONS DAILY, SIZ...","[76 INVERKIP ROAD, 70 - 74 INVERKIP ROAD, 78 I...","[181, 1519248, 530]",530.0,False,None,None,NaN
113946,11215147,THE GATE INN CO.,NaN,NaN,THE GATE INN THE DELVES,SWANWICK,ALFRETON,DERBYSHIRE,ENGLAND,DE551AQ,...,56302,True,[THE GATE INN],[THE DELVES SWANWICK ALFRETON DERBYSHIRE],[729],729.0,None,None,None,Direct_Name
25473,16519529,CLIFFINN,NaN,NaN,THE CLIFF INN,TOWN END,CRICH,DERBYSHIRE,UNITED KINGDOM,DE45DP,...,56302,True,"[CLIFF INN, CRICH TRAMWAY VILLAGE, NATIONAL TR...","[TOWN END CRICH MATLOCK DERBYSHIRE, CRICH TRAM...","[748, 749, 1776053, 1811891]",748.0,None,None,None,Direct_Name
59491,16379120,KASHMIR 1,NaN,NaN,1 ALBION STREET,NaN,CHELTENHAM,NaN,ENGLAND,GL522LH,...,56101,True,[KASHMIR RESTAURANT],[1 ALBION STREET],[1379],1379.0,True,None,None,NaN
104053,SC531326,SPICE GARDEN STIRLING,NaN,NaN,23 ALLAN PARK,NaN,STIRLING,NaN,SCOTLAND,FK82LT,...,56101,True,"[CARLTON BINGO CLUB, DESTINY CHURCH, SPICE GAR...","[28 ALLAN PARK, 28 ALLAN PARK, 23 ALLAN PARK]","[1581, 1585, 1580]",1580.0,False,None,None,NaN
121993,08589456,TUMAN CATERING,NaN,NaN,15 BYWOOD AVENUE,NaN,CROYDON,NaN,NaN,CR07RB,...,56103,True,"[BYWOOD DELI, BYWOOD KEBAB HOUSE]","[17 BYWOOD AVENUE, 15 BYWOOD AVENUE]","[1381780, 1661]",1661.0,False,None,None,NaN
93701,15478214,RIZGAR,NaN,NaN,107 BRIGHTON ROAD,NaN,COULSDON,NaN,ENGLAND,CR52NG,...,56103,True,"[CHAI CAFE, DOMINO'S PIZZA, FLAMINGO KEBAB HOU...","[109 BRIGHTON ROAD, 119 BRIGHTON ROAD, 107 BRI...","[1636848, 998236, 1680, 1613295, 1551472, 1726...",1680.0,False,None,None,NaN


In [ ]:
hospitality_df[hospitality_df['Is_Latest_Match']==False]['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active - Proposal to Strike off,510
Active,38
Liquidation,11


In [ ]:
hospitality_df[hospitality_df['Is_Latest_Match']!=False]['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,24002
Active - Proposal to Strike off,1235
Liquidation,29
Voluntary Arrangement,2


In [ ]:
hospitality_df = hospitality_df[hospitality_df['Is_Latest_Match']!=False]

In [ ]:
len(hospitality_df)

25268

In [ ]:
hospitality_df['CompanyStatus'].value_counts()

,count
CompanyStatus,
Active,24002
Active - Proposal to Strike off,1235
Liquidation,29
Voluntary Arrangement,2


In [ ]:
hospitality_df.head()

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,SIC_Numeric,PostCode_Validity,Potential_matches_name,Potential_matches_address,Potential_matches_businessid,Final_Business_ID,Single_remove,MatchScore,Is_Latest_Match,Match_Method
68397,SC780693,M&S MAZZONI,NaN,NaN,6 BAY STREET,NaN,PORT GLASGOW,NaN,SCOTLAND,PA145ED,...,56103,True,[LOUIS'S],[6 BAY STREET],[40],40.0,True,None,None,NaN
65403,SC402065,LIGHTHOUSE BAR,NaN,NaN,37 CATHCART STREET,NaN,GREENOCK,RENFREWSHIRE,SCOTLAND,PA151DG,...,56302,True,"[CHINA GARDEN, FISHERMANS DOCK, LIGHTHOUSE BAR]","[47 - 49 CATHCART STREET, 43 CATHCART STREET, ...","[1551791, 1786883, 113]",113.0,False,None,None,NaN
93534,SC444940,RISTORANTE ARLECCHINO,NaN,NaN,68 WEST BLACKHALL STREET,NaN,GREENOCK,NaN,SCOTLAND,PA151XG,...,56101,True,[L'ARLECCHINO RESTAURANT],[68 WEST BLACKHALL STREET],[299],299.0,True,None,None,NaN
3003,SC857241,AF EATS,NaN,NaN,78 INVERKIP ROAD,NaN,GREENOCK,NaN,SCOTLAND,PA169AP,...,56103,True,"[BARRS COTTAGE CHIP SHOP, MORRISONS DAILY, SIZ...","[76 INVERKIP ROAD, 70 - 74 INVERKIP ROAD, 78 I...","[181, 1519248, 530]",530.0,False,None,None,NaN
113946,11215147,THE GATE INN CO.,NaN,NaN,THE GATE INN THE DELVES,SWANWICK,ALFRETON,DERBYSHIRE,ENGLAND,DE551AQ,...,56302,True,[THE GATE INN],[THE DELVES SWANWICK ALFRETON DERBYSHIRE],[729],729.0,None,None,None,Direct_Name


In [ ]:
hospitality_df = hospitality_df.merge(
    ratings_df,
    how='left',
    left_on='Final_Business_ID',
    right_on='FHRSID'
)

In [ ]:
hospitality_df.head()

,CompanyNumber,CompanyName,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,Longitude,NewRatingPending,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,Structural,PostCode_Validity_y
0,SC780693,M&S MAZZONI,NaN,NaN,6 BAY STREET,NaN,PORT GLASGOW,NaN,SCOTLAND,PA145ED,...,-4.687872,False,PA145ED,2025-06-18,fhis_pass_en-GB,Pass,NaN,FHIS,NaN,True
1,SC402065,LIGHTHOUSE BAR,NaN,NaN,37 CATHCART STREET,NaN,GREENOCK,RENFREWSHIRE,SCOTLAND,PA151DG,...,-4.751259,False,PA151DG,2025-03-21,fhis_pass_en-GB,Pass,NaN,FHIS,NaN,True
2,SC444940,RISTORANTE ARLECCHINO,NaN,NaN,68 WEST BLACKHALL STREET,NaN,GREENOCK,NaN,SCOTLAND,PA151XG,...,-4.761536,False,PA151XG,2024-12-09,fhis_pass_en-GB,Pass,NaN,FHIS,NaN,True
3,SC857241,AF EATS,NaN,NaN,78 INVERKIP ROAD,NaN,GREENOCK,NaN,SCOTLAND,PA169AP,...,-4.786099,False,PA169AP,2025-10-02,fhis_pass_en-GB,Pass,NaN,FHIS,NaN,True
4,11215147,THE GATE INN CO.,NaN,NaN,THE GATE INN THE DELVES,SWANWICK,ALFRETON,DERBYSHIRE,ENGLAND,DE551AQ,...,-1.390962,False,DE551AQ,2024-11-05,fhrs_4_en-GB,4,NaN,FHRS,5.0,True


In [ ]:
hospitality_df.to_excel('Final Dataset.xlsx',index = False)